In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
# Introduce 'vacation' requests (not covered days off)
# Validation routine

def validate_vacation_capacity(staff_vacations, start_date, total_weeks, min_required_staff=3):
    """
    Ensures that on any given day, there are enough workers available to fill the 
    Day shift, Night shift, and account for the 24-hour post-night rest worker.
    """
    print("Checking vacation lists for staffing bottlenecks...")
    bottlenecks_found = False
    total_days = total_weeks * 7
    
    for day_idx in range(total_days):
        current_date = start_date + timedelta(days=day_idx)
        date_str = current_date.strftime("%Y-%m-%d")
        
        on_vacation = [worker for worker, dates in staff_vacations.items() if date_str in dates]
        available_count = len(staff_vacations) - len(on_vacation)
        
        if available_count < min_required_staff:
            print(f"⚠️  CRITICAL BOTTLENECK on {date_str} ({current_date.strftime('%A')}):")
            print(f"    Only {available_count} workers available! Minimum required is {min_required_staff}.")
            print(f"    Workers on vacation: {', '.join(on_vacation)}\n")
            bottlenecks_found = True
            
    if bottlenecks_found:
        print("❌ Schedule generation halted. Please adjust conflicting vacation requests.")
        return False
    
    print("✅ No bottlenecks found! All dates have sufficient staffing levels.\n")
    return True


def generate_exact_40hr_vacation_schedule(start_date_str, total_weeks=26):
    # 1. Define your actual staff and their exact vacation dates (YYYY-MM-DD)
    # If a worker has no vacation, use set()
    staff_vacations = {
        "Alice": {"2026-09-24", "2026-09-25", "2026-12-25"},
        "Bob": set(),
        "Charlie": {"2026-10-01", "2026-10-02"},
        "Diana": set(),
        "Evan": {"2026-12-24", "2026-12-25", "2026-12-26"},
        "Fiona": set(),
        "George": set(),
        "Hannah": {"2026-11-26"}  # Thanksgiving
    }
    
    workers = list(staff_vacations.keys())
    
    if len(workers) != 8:
        raise ValueError(f"Schedule requires exactly 8 workers. Found {len(workers)} in staff_vacations.")

    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    
    # Run pre-flight capacity guardrail
    if not validate_vacation_capacity(staff_vacations, start_date, total_weeks):
        return

    # Cumulative global trackers for the 6 months
    global_counts = {
        w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Full_Clinic': 0, 'Half_Clinic': 0, 'Total_Hours': 0} 
        for w in workers
    }
    
    schedule_data = []
    last_night_worker = None

    # Process schedule week-by-week to lock down the 40-hour rule
    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        weekly_hospital_counts = {w: 0 for w in workers}
        week_days_manifest = {}
        
        # --- PHASE 1: Assign 12-Hour Hospital Shifts (Mon - Sun) ---
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {
                'day_name': day_name, 'day_hospital': None, 'night_hospital': None, 
                'full_clinic_staff': [], 'half_clinic_staff': []
            }
            
            # Filter out anyone currently on vacation today
            active_today = [w for w in workers if date_str not in staff_vacations[w]]
            
            # --- Day Shift Assignment ---
            available_for_day = [w for w in active_today if w != last_night_worker and weekly_hospital_counts[w] < 3]
            available_for_day.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Day_Shifts'], global_counts[w]['Total_Hours']))
            assigned_day_worker = available_for_day[0]
            
            week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
            weekly_hospital_counts[assigned_day_worker] += 1
            global_counts[assigned_day_worker]['Day_Shifts'] += 1
            global_counts[assigned_day_worker]['Total_Hours'] += 12
            
            # --- Night Shift Assignment ---
            available_for_night = [
                w for w in active_today 
                if w != assigned_day_worker and w != last_night_worker and weekly_hospital_counts[w] < 3
            ]
            available_for_night.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Night_Shifts'], global_counts[w]['Total_Hours']))
            assigned_night_worker = available_for_night[0]
            
            week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
            weekly_hospital_counts[assigned_night_worker] += 1
            global_counts[assigned_night_worker]['Night_Shifts'] += 1
            global_counts[assigned_night_worker]['Total_Hours'] += 12
            
            last_night_worker = assigned_night_worker

        # --- PHASE 2: Backfill Clinic Requirements (Mon - Fri Only) ---
        for worker in workers:
            shifts = weekly_hospital_counts[worker]
            
            # Exact combination math to equal 40 hours for the week
            if shifts == 0:   req_full, req_half = 5, 0
            elif shifts == 1: req_full, req_half = 3, 1
            elif shifts == 2: req_full, req_half = 2, 0
            elif shifts == 3: req_full, req_half = 0, 1
            else:             req_full, req_half = 0, 0
            
            full_assigned = 0
            half_assigned = 0
            
            for date_str, day_data in week_days_manifest.items():
                if full_assigned == req_full and half_assigned == req_half:
                    break
                
                # Rule: Clinics are strictly closed on weekends or if the worker is on vacation
                if day_data['day_name'] in ['Saturday', 'Sunday'] or date_str in staff_vacations[worker]:
                    continue
                
                # Rule: Conflict & 24hr Post-Night Rest Checks
                is_on_day_hospital = day_data['day_hospital'] == worker
                is_on_night_hospital = day_data['night_hospital'] == worker
                
                prev_date = datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)
                prev_date_str = prev_date.strftime("%Y-%m-%d")
                was_resting = False
                if prev_date_str in week_days_manifest:
                    was_resting = week_days_manifest[prev_date_str]['night_hospital'] == worker
                
                # If they pass all rules, assign clinic blocks
                if not is_on_day_hospital and not is_on_night_hospital and not was_resting:
                    if full_assigned < req_full:
                        day_data['full_clinic_staff'].append(worker)
                        full_assigned += 1
                        global_counts[worker]['Full_Clinic'] += 1
                        global_counts[worker]['Total_Hours'] += 8
                    elif half_assigned < req_half:
                        day_data['half_clinic_staff'].append(worker)
                        half_assigned += 1
                        global_counts[worker]['Half_Clinic'] += 1
                        global_counts[worker]['Total_Hours'] += 4

        # --- PHASE 3: Compile Manifest Chronologically ---
        for date_str, day_data in sorted(week_days_manifest.items()):
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'],
                'Day Shift (12hr)': day_data['day_hospital'],
                'Night Shift (12hr)': day_data['night_hospital'],
                'Full Clinic (8hr)': ", ".join(day_data['full_clinic_staff']) if day_data['full_clinic_staff'] else "None",
                'Half Clinic (4hr)': ", ".join(day_data['half_clinic_staff']) if day_data['half_clinic_staff'] else "None"
            })

    # 4. Export results
    csv_filename = "hospital_exact_40hr_vacation_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        
    print(f"Schedule successfully compiled and exported to {csv_filename}!\n")
    print("6-Month Cumulative Totals (Vacations Deducted, Equity Maintained):")
    print(f"{'Worker Name':<22} | {'Day Shifts':<10} | {'Night Calls':<11} | {'Full Clinic':<11} | {'Half Clinic':<11} | {'Total Hours':<12}")
    print("-" * 92)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<22} | {counts['Day_Shifts']:<10} | {counts['Night_Shifts']:<11} | {counts['Full_Clinic']:<11} | {counts['Half_Clinic']:<11} | {counts['Total_Hours']:<12} hrs")

# Run starting tomorrow (Monday, September 14, 2026)
generate_exact_40hr_vacation_schedule("2026-09-20")
